### Structured Responses
Structured responses let you enforce predictable, schema‑driven outputs (via StructuredOutputParser, Pydantic models, or JSON mode), making LLMs behave more like APIs than chatbots.

We will implement all 4 methods
1. TypeDict
2. Annotated TypeDict
3. Pydantic
4. JSON schema


In [2]:
%pip install langchain-groq

  Using cached langchain_groq-1.1.3-py3-none-any.whl.metadata (2.9 kB)
  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
Using cached langchain_groq-1.1.3-py3-none-any.whl (20 kB)
Using cached groq-0.37.1-py3-none-any.whl (137 kB)

  Attempting uninstall: groq

    Found existing installation: groq 1.4.0

   ---------------------------------------- 0/2 [groq]
    Uninstalling groq-1.4.0:
   ---------------------------------------- 0/2 [groq]
      Successfully uninstalled groq-1.4.0
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---

### TypeDict

In [4]:
from langchain_groq import ChatGroq
from typing import TypedDict
import os

api_key = os.getenv("GROQ_API_KEY")
model=ChatGroq(model_name="llama-3.3-70b-versatile",groq_api_key=api_key)

#schema
class Review(TypedDict):
    summary:str
    sentiment:str

prompt="""The hardware is great, but the software feels bloateed.
There are too many pre-installed apps that I never use and can't uninstall.
The battery life is decent, but it could be better.
Also the UI lools outdated compared to other brands.
Hoping for a software update to fix this. Overall, it's an average phone with some
good features but also some drawbacks."""


structured_model=model.with_structured_output(Review)
structured_response=structured_model.invoke(prompt)
print(structured_response)

{'sentiment': 'neutral', 'summary': 'The phone has great hardware but is let down by bloated software, outdated UI, and limited ability to uninstall pre-installed apps, resulting in an average overall experience.'}


#### Annotated TypeDict

In [5]:
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated,Optional
import os
 

api_key = os.getenv("GROQ_API_KEY")
model=ChatGroq(model_name="llama-3.3-70b-versatile",groq_api_key=api_key)


class Schema(TypedDict):
    key_themes:Annotated[list[str],"must write all the key themes discussed in the review in a list"]
    summary:Annotated[str,"must write a brief summary of the review"]
    sentiment:Annotated[str,"must return the sentiment of the review,either positive or Negative"]
    pros:Annotated[Optional[list[str]],"write adown all the pros inside a list"] ## this is optional 
    cons:Annotated[Optional[list[str]],"write adown all the cons inside a list"] ## this is optional 
    
structured_model=model.with_structured_output(Schema)

prompt="""I've been using this phone for about a month, and my experience has been mixed. The display is excellent—bright, smooth, and great for watching videos outdoors. Battery life is another strong point; I can easily get through a full day of heavy use without worrying about charging.

The camera performs well in daylight and captures detailed photos. The design also feels premium for the price, and the phone is comfortable to hold despite its large screen.

However, the software experience is disappointing. There are too many pre-installed apps, and I keep getting notifications from features I never asked for. The interface feels cluttered, and occasional lag makes the phone feel slower than the hardware should allow.

The speakers are average, and low-light camera performance could be better. I also noticed that some apps take longer than expected to open after prolonged use.

Overall, this phone offers great hardware, an excellent display, and strong battery life, but the software experience holds it back. If the manufacturer reduces the bloatware and improves optimization, it would be a much easier recommendation."""

response=structured_model.invoke(prompt)
print(response)

{'cons': ['disappointing software experience', 'too many pre-installed apps', 'cluttered interface', 'occasional lag', 'average speakers', 'poor low-light camera performance'], 'key_themes': ['display', 'battery life', 'camera performance', 'software experience', 'design', 'bloatware'], 'pros': ['excellent display', 'strong battery life', 'premium design', 'detailed daylight photos'], 'sentiment': 'Mixed', 'summary': 'The phone has great hardware, excellent display, and strong battery life, but is held back by a disappointing software experience with too many pre-installed apps, cluttered interface, and occasional lag'}


#### Pydantic

In [6]:
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated,Optional,Literal
from pydantic import BaseModel,Field
import os

api_key = os.getenv("GROQ_API_KEY")
model=ChatGroq(model_name="llama-3.3-70b-versatile",groq_api_key=api_key)

#schema
class Schema(BaseModel):
    key_themes: list[str] = Field(
        description="Must write all key themes discussed in the review"
    )

    summary: str = Field(
        description="Must write a brief summary of the review"
    )

    sentiment: str = Field(
        description="Must return the sentiment of the review"
    )

    pros: Optional[list[str]] = Field(
        default=None,
        description="Write down all the pros inside a list"
    )

    cons: Optional[list[str]] = Field(
        default=None,
        description="Write down all the cons inside a list"
    )
    name:Optional[str]=Field(
        default=None,
        description="Write down the name of the person who wrote the review,generally found at the nd of the review"
    )

structured_model=model.with_structured_output(Schema,strict=True)

prompt="""I've been using this phone for about a month, and my experience has been mixed. The display is excellent—bright, smooth, and great for watching videos outdoors. Battery life is another strong point; I can easily get through a full day of heavy use without worrying about charging.

The camera performs well in daylight and captures detailed photos. The design also feels premium for the price, and the phone is comfortable to hold despite its large screen.

However, the software experience is disappointing. There are too many pre-installed apps, and I keep getting notifications from features I never asked for. The interface feels cluttered, and occasional lag makes the phone feel slower than the hardware should allow.

The speakers are average, and low-light camera performance could be better. I also noticed that some apps take longer than expected to open after prolonged use.

Overall, this phone offers great hardware, an excellent display, and strong battery life, but the software experience holds it back. If the manufacturer reduces the bloatware and improves optimization, it would be a much easier recommendation.
BY-Gupta"""

response=structured_model.invoke(prompt)
print(response)

key_themes=['display', 'battery life', 'camera', 'design', 'software experience', 'bloatware'] summary='The phone has great hardware, display, and battery life but is held back by a disappointing software experience with too many pre-installed apps and occasional lag' sentiment='neutral' pros=['excellent display', 'strong battery life', 'premium design'] cons=['disappointing software experience', 'too many pre-installed apps', 'average speakers', 'low-light camera performance could be better'] name='BY-Gupta'


#### Json Schema

In [7]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated,Optional,Literal
import os

api_key = os.getenv("GROQ_API_KEY")
model=ChatGroq(model_name="llama-3.3-70b-versatile",groq_api_key=api_key)

#json schema

json_schema = {
    "title": "Review",
    "type": "object",
    "properties": {
        "key_themes": {
            "type": "array",
            "items": {
                "type": "string"
            },
            "description": "Write down all the key themes discussed in the review in a list"
        },
        "summary": {
            "type": "string",
            "description": "Write a brief summary of the review"
        },
        "sentiment": {
            "type": "string",
            "enum":["pos","neg"],
            "description": "Return the sentiment of the review, either Positive or Negative"
        },
        "pros": {
            "type": ["array", "null"],
            "items": {
                "type": "string"
            },
            "description": "Write down all the pros inside a list"
        },
        "cons": {
            "type": ["array", "null"],
            "items": {
                "type": "string"
            },
            "description": "Write down all the cons inside a list"
        }
    },
    "required": [
        "key_themes",
        "summary",
        "sentiment"
    ]
}

structured_model=model.with_structured_output(json_schema,strict=True)

prompt="""I've been using this phone for about a month, and my experience has been mixed. The display is excellent—bright, smooth, and great for watching videos outdoors. Battery life is another strong point; I can easily get through a full day of heavy use without worrying about charging.
The camera performs well in daylight and captures detailed photos. The design also feels premium for the price, and the phone is comfortable to hold despite its large screen.
However, the software experience is disappointing. There are too many pre-installed apps, and I keep getting notifications from features I never asked for. The interface feels cluttered, and occasional lag makes the phone feel slower than the hardware should allow.
The speakers are average, and low-light camera performance could be better. I also noticed that some apps take longer than expected to open after prolonged use.
Overall, this phone offers great hardware, an excellent display, and strong battery life, but the software experience holds it back. If the manufacturer reduces the bloatware and improves optimization, it would be a much easier recommendation.
BY-Gupta"""

response=structured_model.invoke(prompt)
print(response)

{'key_themes': ['display', 'battery life', 'camera', 'design', 'software experience'], 'sentiment': 'neg', 'summary': 'The phone has great hardware, excellent display, and strong battery life, but is held back by a disappointing software experience with too many pre-installed apps and occasional lag'}


### Output Parsers

Output parsers are the bridge between free‑form LLM text and structured, validated data you can use in code.

#### str Output Parsers

In [8]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

api_key = os.getenv("GROQ_API_KEY")
model=ChatGroq(model_name="llama-3.3-70b-versatile",groq_api_key=api_key)

# 1st prompt template
template1=PromptTemplate(template="Write a detailed report on {topic}",
                         input_variables=['topic']
)

# 2nd prompt template
template2=PromptTemplate(template="Write a 5 line summary on the following {text}",
                         input_variables=['text']
)

parser=StrOutputParser()

chain=template1 | model | parser | template2 | model | parser
response=chain.invoke({'topic':"Black Hole"})
print(response)

Here is a 5-line summary of the introduction to black holes:
A black hole is a region in space with an incredibly strong gravitational pull that nothing can escape. 
Black holes are formed when a massive star collapses in on itself, causing its gravity to warp spacetime. 
The study of black holes has led to a deeper understanding of the laws of physics and the behavior of matter in extreme conditions. 
These mysterious objects continue to fascinate scientists and the public alike, with ongoing research revealing new insights into the universe. 
Through their unique properties and behaviors, black holes offer a window into the extreme conditions of the universe, driving further exploration and discovery.
